In [ ]:
# | default_exp core

In [ ]:
# | hide
from nbdev.showdoc import *

In [ ]:
# | export
import json
import asyncio
import os


async def search_code(query: str, k: int = 5) -> list[dict]:
    """Search the codebase semantically using ColGrep."""
    proc = await asyncio.create_subprocess_exec(
        "colgrep",
        "-e",
        "^def |^class ",
        query,
        "-k",
        str(k),
        "--json",
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
        cwd=os.environ.get("HYPERRUN_REPO_PATH"),
    )
    stdout, _ = await proc.communicate()
    return json.loads(stdout)

In [ ]:
# | export
from pathlib import Path


async def ensure_index():
    repo = os.environ.get("HYPERRUN_REPO_PATH", ".")
    idx = Path(repo) / ".colgrep"
    if not idx.exists():
        proc = await asyncio.create_subprocess_exec("colgrep", "init", cwd=repo)
        await proc.wait()


In [ ]:
# | hide
await search_code("insert article")

[{'unit': {'name': 'filter_site',
   'qualified_name': 'gsc_storage.py::filter_site',
   'file': '/home/kobo/Desktop/seo_rat/seo_rat/gsc_storage.py',
   'line': 20,
   'end_line': 21,
   'language': 'python',
   'unit_type': 'function',
   'signature': 'def filter_site(query, site_url: str):',
   'docstring': None,
   'parameters': ['query', 'site_url'],
   'return_type': None,
   'extends': None,
   'parent_class': None,
   'calls': ['where'],
   'called_by': [],
   'complexity': 1,
   'has_loops': False,
   'has_branches': False,
   'has_error_handling': False,
   'variables': [],
   'imports': [],
   'code': 'def filter_site(query, site_url: str):\n    return query.where(GSCAnalytics.site_url == site_url)'},
  'score': 6.9994364},
 {'unit': {'name': 'filter_dimension',
   'qualified_name': 'gsc_storage.py::filter_dimension',
   'file': '/home/kobo/Desktop/seo_rat/seo_rat/gsc_storage.py',
   'line': 29,
   'end_line': 30,
   'language': 'python',
   'unit_type': 'function',
   'signa

In [ ]:
# | export
from lisette import AsyncChat
from dotenv import load_dotenv

load_dotenv()
sp = """You have a `search_code` tool that searches a Python codebase using ColGrep (semantic code search).
It uses hybrid search: regex pre-filters to functions/classes, then ranks semantically.
- Increase `k` if you need more results.
- Scores are more accurate when functions have rich docstrings.
- You can call the tool multiple times with different queries to find related code."""

chat = AsyncChat("claude-sonnet-4-20250514", sp=sp, tools=[search_code])

In [ ]:
# | hide
res = await chat("What functions handle articles?", stream=True)
async for o in res:
    if (
        hasattr(o, "choices")
        and hasattr(o.choices[0], "delta")
        and o.choices[0].delta.content
    ):
        print(o.choices[0].delta.content, end="", flush=True)


I'll search the codebase for functions that handle articles.Based on my search for functions that handle articles, I found several relevant functions in the codebase, though they appear to be more focused on SEO analysis and page content rather than traditional "article" management. Here are the key functions:

## Main Article/Page Handling Functions:

1. **`full_page_audit`** (page_seo_audit.py) - The primary function for comprehensive SEO analysis of individual pages/articles:
   - Analyzes content, metadata, headers, images
   - Performs GSC (Google Search Console) performance analysis
   - Classifies query intents and detects trends
   - Finds missing queries and green keywords

2. **`get_page_analytics`** (gsc_storage.py) - Retrieves analytics data for specific pages:
   - Gets clicks, impressions, position, and CTR data
   - Extracts top queries for a page

3. **`classify_page_intents`** (gsc_insights.py) - Analyzes search intent for page queries:
   - Classifies the intent behin

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()